In [8]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

In [9]:
def display_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    fig = go.Figure()

    cm = confusion_matrix(y_true, y_pred)
    cm_percentages = cm / cm.sum(axis=1, keepdims=True) * 100
    accuracy = (y_true == y_pred).mean() * 100
    labels = list(map(str, np.unique(y_true)))

    fig.add_trace(
        go.Heatmap(
            z=cm_percentages,
            x=labels,
            y=labels,
            colorscale="Darkmint",
            showscale=True,
            texttemplate="%{text}",
            text=[
                [cm[i, j] for j in range(cm.shape[1])]
                for i in range(cm.shape[0])
            ],
        )
    )

    fig.update_layout(
        title={
            "text": f"{title}<br>Accuracy: {accuracy:.2f}%",
            "x": 0.5,
        },
        xaxis_title="Predicted",
        yaxis_title="True",
        width=500,
        height=500,
    )
    
    fig.show()

In [10]:
df = pd.read_csv('Data/wine.csv', header=None)
df.columns = [
    "Label", "Alcohol", "Malic acid", "Ash", "Alcalinity of ash", "Magnesium",
    "Total phenols", "Flavanoids", "Nonflavanoid phenols", "Proanthocyanins",
    "Color intensity", "Hue", "OD280/OD315 of diluted wines", "Proline"
]

df.head()

,Label,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


In [16]:
features = df.iloc[:, 1:].to_numpy()
labels = df.iloc[:, 0].to_numpy()

L2 = lambda x, y: np.sqrt(np.sum(np.square(x - y)))
NN = lambda x, X, Y: Y[np.argmin([L2(x, sample) for sample in X])]

In [17]:
# Part 1

drop = lambda x, i: np.delete(x, i, axis=0)
LOOCV = lambda x, i: NN(x, drop(features, i), drop(labels, i))

predictions = np.array([LOOCV(x, i) for i, x in enumerate(features)])
display_confusion_matrix(labels, predictions, title="LOOCV Confusion Matrix")

In [18]:
# Part 2

ks = np.unique(np.round(np.linspace(2, 100, 20)).astype(int))
accuracies = []

for k in ks:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    preds, truths = [], []

    for train_index, test_index in kf.split(features):
        train_X, test_X = features[train_index], features[test_index]
        train_y, test_y = labels[train_index], labels[test_index]

        for sample, true_label in zip(test_X, test_y):
            pred_label = NN(sample, train_X, train_y)
            preds.append(pred_label)
            truths.append(true_label)

    acc = accuracy_score(truths, preds)
    accuracies.append(acc)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ks, y=accuracies))
fig.update_layout(
    title="Accuracy vs. k",
    xaxis_title="k",
    yaxis_title="Accuracy",
    width=800,
    height=400,
)
fig.show()

In [19]:
# Part 3

nfeatures = (features - features.min(axis=0)) / (features.max(axis=0) - features.min(axis=0))
NLOOCV = lambda x, i: NN(x, drop(nfeatures, i), drop(labels, i))

predictions = np.array([NLOOCV(x, i) for i, x in enumerate(nfeatures)])
display_confusion_matrix(labels, predictions, title="Normalized LOOCV Confusion Matrix")